## Setup
Load libraries and create experiment folder.

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score
from FIT_python.config import SPLITS_DIR, RESULTS_DIR, RESULTS_DATA_DIR

EXP_DIR = Path('experiments/baseline_analysis')
EXP_DIR.mkdir(parents=True, exist_ok=True)


## Create Splits
Use the provided SplitWrapper to regenerate train/test splits.

In [ ]:

from FIT_python.data_split_and_summary.datasplit_and_summary_wraper import SplitWrapper

SplitWrapper().split_all(as_csv=False)


## Train simple sex model
Small hyperparameter search on the existing splits.

In [ ]:

from FIT_python.pipeline_sex.pipeline_wrapper_sex import get_pipeline_steps
from FIT_python.pipeline_sex.models import MODELS
from pathlib import Path
from sklearn.model_selection import cross_val_score, PredefinedSplit

sp_dir = SPLITS_DIR / 'eurasian_otter'
train_df = pd.read_parquet(sp_dir / 'train.parquet')
test_df = pd.read_parquet(sp_dir / 'test.parquet')

num_cols = train_df.select_dtypes(include=np.number).columns
feature_cols = [c for c in num_cols if c != 'Fold']
X_train = train_df[feature_cols]
y_train = train_df['sex'].map({'f':0,'m':1})
folds = PredefinedSplit(train_df['Fold']) if 'Fold' in train_df.columns else 3
steps = get_pipeline_steps(fs_method='forward', fs_k=10, scaler_method='standard')
steps.append(('classifier', MODELS['logreg_l2']))
pipe = Pipeline(steps)
cv_acc = cross_val_score(pipe, X_train, y_train, cv=folds, scoring='balanced_accuracy')
pipe.fit(X_train, y_train)
X_test = test_df[feature_cols]
y_test = test_df['sex'].map({'f':0,'m':1})
preds = pipe.predict(X_test)
sex_bal_acc = balanced_accuracy_score(y_test, preds)
sex_results = pd.DataFrame({'true': test_df['sex'], 'pred': preds})
sex_results.to_csv(EXP_DIR / 'sex_results.csv', index=False)


## Baseline individual ID distances

In [ ]:

from FIT_python.pipeline_individual_id.generate_trails_and_trailpairs import generate_pairwise_comparisons_from_df
from FIT_python.pipeline_individual_id.baseline_pipeline import DistanceBaseline

all_df = pd.concat([train_df, test_df], ignore_index=True)
comparisons, summary = generate_pairwise_comparisons_from_df(all_df, num_individuals=3, window_lengths=[5], n_windows=1)
base_df = DistanceBaseline.run(train_df, comparisons, feature_cols, val_df=test_df, reducers=['lda'], scaler_methods='standard', n_jobs=1)
base_df.to_csv(EXP_DIR / 'baseline_results.csv', index=False)
summary.to_csv(EXP_DIR / 'baseline_summary.csv', index=False)


## Pipeline overview

In [ ]:

import networkx as nx
G = nx.DiGraph()
G.add_edges_from([('Data Import','Feature Engineering'),('Feature Engineering','Outlier/Scaling'),('Outlier/Scaling','Feature Selection'),('Feature Selection','Dimensionality Reduction'),('Dimensionality Reduction','Modeling'),('Modeling','Evaluation'),('Evaluation','Visualisation')])
pos = nx.spring_layout(G, seed=42)
plt.figure(figsize=(8,5))
nx.draw_networkx_nodes(G,pos,node_size=1200,node_color='#4C72B0')
nx.draw_networkx_edges(G,pos,arrowstyle='->',arrowsize=15)
nx.draw_networkx_labels(G,pos,font_color='white')
plt.axis('off')
plt.title('Pipeline Overview')
plt.show()


## Feature frequency

In [ ]:

selected = getattr(pipe.named_steps.get('select'),'selected_features_',[])
count_df = pd.DataFrame({'feature':selected})
feature_counts = count_df['feature'].value_counts()
plt.figure(figsize=(8,4))
sns.barplot(y=feature_counts.index, x=feature_counts.values, color='#4C72B0')
plt.xlabel('Times Selected')
plt.ylabel('Feature')
plt.title('Most Frequently Selected Features')
plt.tight_layout()
plt.show()


## Distance metric comparison

In [ ]:

melted = base_df.melt(id_vars=['reducer'], value_vars=[c for c in base_df.columns if 'euclidean' in c or 'cosine' in c or 'manhattan' in c], var_name='metric', value_name='distance')
plt.figure(figsize=(8,4))
sns.boxplot(data=melted, x='metric', y='distance')
plt.ylabel('Distance')
plt.xlabel('Metric')
plt.title('Distance Metrics Distribution')
plt.tight_layout()
plt.show()


## Embeddings 2D

In [ ]:

from sklearn.decomposition import PCA
X_emb = PCA(n_components=2).fit_transform(X_train)
plt.scatter(X_emb[:,0], X_emb[:,1], c=y_train, cmap='tab10', s=20)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA Embedding')
plt.tight_layout()
plt.show()


## Cumulative training times

In [ ]:

from time import perf_counter
steps = get_pipeline_steps(fs_method='forward', fs_k=10, scaler_method='standard')
steps.append(('classifier', MODELS['logreg_l2']))
pipe_tmp = Pipeline(steps)
X_tmp = X_train.copy()
timings = []
for name, step in pipe_tmp.steps[:-1]:
    t0 = perf_counter();
    X_tmp = step.fit_transform(X_tmp, y_train) if hasattr(step,'fit_transform') else step.fit(X_tmp, y_train).transform(X_tmp);
    timings.append({'step':name,'seconds':perf_counter()-t0})
t0=perf_counter();
pipe_tmp.steps[-1][1].fit(X_tmp,y_train);
timings.append({'step':'classifier','seconds':perf_counter()-t0})
timing_df = pd.DataFrame(timings)
timing_df['cumulative']=timing_df['seconds'].cumsum()
plt.plot(timing_df['step'], timing_df['cumulative'], marker='o')
plt.ylabel('Cumulative Seconds')
plt.xlabel('Pipeline Step')
plt.xticks(rotation=45, ha='right')
plt.title('Cumulative Training Times')
plt.tight_layout()
plt.show()


## Hyperparameter heatmap

In [ ]:

param_results = []
for fs in ['forward','variance']:
    steps = get_pipeline_steps(fs_method=fs, fs_k=5, scaler_method='standard')
    steps.append(('classifier', MODELS['logreg_l2']))
    pipe = Pipeline(steps)
    score = cross_val_score(pipe, X_train, y_train, cv=folds, scoring='balanced_accuracy').mean()
    param_results.append({'fs_method':fs,'reduce_pre_method':None,'cv_balanced_accuracy':score})
hyper_df = pd.DataFrame(param_results)
pivot = hyper_df.pivot_table(index='fs_method', columns='reduce_pre_method', values='cv_balanced_accuracy')
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis', vmin=0, vmax=1)
plt.xlabel('Pre-dim. Reducer')
plt.ylabel('Feature Selector')
plt.title('Mean CV Accuracy')
plt.tight_layout()
plt.show()


## Prediction quality heatmaps

In [ ]:

from FIT_python.pipeline_sex.sex_predict_and_visualisation import plot_quality_heatmaps
sex_results_full = test_df.copy(); sex_results_full['pred_best_sex']=preds; sex_results_full['pred_best_proba_f']=pipe.predict_proba(X_test)[:,0]; sex_results_full['pred_best_proba_m']=pipe.predict_proba(X_test)[:,1];
plot_quality_heatmaps(sex_results_full, pred_col='pred_best_sex', proba_cols=['pred_best_proba_f','pred_best_proba_m'])


## Split overview

In [ ]:

from FIT_python.data_split_and_summary.summary_data_wrapper import SummaryWrapper
summary_csv = EXP_DIR / 'split_summary.csv'
SummaryWrapper().summarize_all(fig_dir=EXP_DIR/ 'summary_figs')
df_summary = pd.read_csv(RESULTS_DATA_DIR / 'summary.csv')
from FIT_python.data_split_and_summary.summary_data_utils import plot_split_proportions
plot_split_proportions(df_summary, EXP_DIR / 'summary_figs')


## Model performance over iterations

In [ ]:

iterations = []
for i in range(3):
    score = cross_val_score(pipe, X_train, y_train, cv=folds, scoring='balanced_accuracy').mean()
    iterations.append({'iteration':i,'balanced_accuracy':score})
iteration_metrics = pd.DataFrame(iterations)
sns.lineplot(data=iteration_metrics, x='iteration', y='balanced_accuracy', marker='o')
plt.xlabel('Iteration')
plt.ylabel('Balanced Accuracy')
plt.title('Model Performance Over Iterations')
plt.tight_layout()
plt.show()


## Interactive gallery

In [ ]:

from ipywidgets import interact, Dropdown
from IPython.display import Image, display
fig_paths = list((EXP_DIR/'summary_figs').glob('*.png'))
def show_image(name):
    display(Image(filename=name))
dropdown = Dropdown(options=[str(p) for p in fig_paths], description='Figure:')
interact(lambda name: show_image(name), name=dropdown)


## Built-in: Confusion matrix

In [ ]:

from FIT_python.pipeline_sex.sex_predict_and_visualisation import plot_confusion
sex_results_full = test_df.copy(); sex_results_full['pred_logreg_l2_sex']=preds
plot_confusion(sex_results_full)


## Built-in: Inference split prediction

In [ ]:

from FIT_python.pipeline_sex.sex_predict_and_visualisation import plot_inference
plot_inference(sex_results_full)


## Built-in: Individual probabilities

In [ ]:

from FIT_python.pipeline_sex.sex_predict_and_visualisation import plot_individual_probabilities
plot_individual_probabilities(sex_results_full, EXP_DIR)


## Built-in: Pipeline timings

In [ ]:

from FIT_python.pipeline_sex.pipeline_wrapper_sex import plot_pipeline_timings
plot_pipeline_timings(timing_df, EXP_DIR)
